# 04 Open-Meteo Wetterdaten abrufen

Dieses Notebook implementiert BD-11 und BD-12. Es liest die in BD-10 erzeugten Geocodes, verbindet sie mit den StatsBomb-Matchdaten und ruft historische Wetterdaten pro eindeutiger Match-Ort-Datum-Kombination von Open-Meteo ab. Anschliessend werden die Wettervariablen fuer die Analyse standardisiert.

Outputs:

- `data/raw/weather/openmeteo_raw.jsonl`
- `data/bronze/weather_observations.parquet`
- `outputs/tables/openmeteo_failures.csv`
- `outputs/tables/openmeteo_weather_imputations.csv`
- `data/silver/weather_features.parquet`
- `outputs/tables/bd12_missing_weather_values.csv`


## Methodischer Ansatz

Die Wetterabfrage wird als reproduzierbarer API-Schritt umgesetzt. Pro eindeutiger Kombination aus Matchdatum und Koordinate wird ein Open-Meteo-Request erzeugt und als JSONL-Record gecacht.

Die API-Antworten werden in strukturierte Match-Level-Beobachtungen ueberfuehrt. Zusaetzliche Zwischenoutputs machen Lookup-Keys, entpackte Stundenwerte und Match-Kandidaten nachvollziehbar, ohne den Notebook-Ablauf aufzublaehen. Im Silver-Schritt werden Analysefelder standardisiert, Temperaturgruppen gebildet und fehlende Werte explizit dokumentiert.

In [1]:
from datetime import datetime, timezone
import json
import time
from urllib.parse import urlencode

import pandas as pd
import requests

from project_paths import project_root as get_project_root
from pipeline_config import OPEN_METEO_ARCHIVE_URL, OPEN_METEO_HOURLY_VARIABLES as HOURLY_VARIABLES, REQUEST_HEADERS

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 30)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
raw_weather_path = base_path / 'data' / 'raw' / 'weather'
weather_cache_path = base_path / 'data' / 'cache' / 'weather'
outputs_tables_path = base_path / 'outputs' / 'tables'

matches_path = bronze_path / 'statsbomb_matches.parquet'
geocodes_path = silver_path / 'venue_geocodes.parquet'
raw_jsonl_path = raw_weather_path / 'openmeteo_raw.jsonl'
weather_output_path = bronze_path / 'weather_observations.parquet'
hourly_weather_output_path = weather_cache_path / 'openmeteo_hourly_weather.parquet'
candidate_rows_output_path = weather_cache_path / 'openmeteo_candidate_rows.parquet'
lookup_keys_output_path = outputs_tables_path / 'openmeteo_lookup_keys.csv'
missing_weather_output_path = outputs_tables_path / 'openmeteo_missing_weather_matches.csv'
weather_imputation_output_path = outputs_tables_path / 'openmeteo_weather_imputations.csv'
failure_output_path = outputs_tables_path / 'openmeteo_failures.csv'

for path in [bronze_path, silver_path, raw_weather_path, weather_cache_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)



## Match-Ort-Datum-Keys vorbereiten

BD-11 soll nicht pro Team-Match oder pro Stadion-Duplikat abfragen, sondern pro eindeutiger Kombination aus Datum, Latitude und Longitude. Deshalb werden die StatsBomb-Matches zuerst mit den BD-10-Geocodes verbunden und danach auf stabile Wetter-Lookup-Keys reduziert.


In [2]:
matches = pd.read_parquet(matches_path)
geocodes = pd.read_parquet(geocodes_path)

required_match_columns = [
    'match_id',
    'match_date',
    'kick_off',
    'short_name',
    'competition_name',
    'season_name',
    'stadium_id',
    'stadium_name',
    'stadium_country_name',
]
required_geocode_columns = [
    'stadium_id',
    'venue_key',
    'location_key',
    'city_name',
    'country_name',
    'latitude',
    'longitude',
    'geocode_status',
    'quality_flag',
    'matched_timezone',
]

missing_match_columns = sorted(set(required_match_columns) - set(matches.columns))
missing_geocode_columns = sorted(set(required_geocode_columns) - set(geocodes.columns))
assert not missing_match_columns, f'Missing columns in StatsBomb matches: {missing_match_columns}'
assert not missing_geocode_columns, f'Missing columns in venue geocodes: {missing_geocode_columns}'

match_locations = matches[required_match_columns].merge(
    geocodes[required_geocode_columns],
    on='stadium_id',
    how='left',
    validate='many_to_one',
)

match_locations['match_date'] = pd.to_datetime(match_locations['match_date']).dt.date.astype(str)
match_locations['kickoff_timestamp'] = pd.to_datetime(
    match_locations['match_date'] + ' ' + match_locations['kick_off'].fillna('12:00:00'),
    errors='coerce',
)
match_locations['latitude_rounded'] = match_locations['latitude'].round(5)
match_locations['longitude_rounded'] = match_locations['longitude'].round(5)
match_locations['weather_lookup_key'] = (
    match_locations['match_date']
    + '__'
    + match_locations['latitude_rounded'].astype(str)
    + '__'
    + match_locations['longitude_rounded'].astype(str)
)

lookup_problem_mask = (
    match_locations['latitude'].isna()
    | match_locations['longitude'].isna()
    | match_locations['match_date'].isna()
    | match_locations['kickoff_timestamp'].isna()
)
lookup_problems = match_locations.loc[lookup_problem_mask].copy()

weather_lookup_keys = (
    match_locations.loc[~lookup_problem_mask, [
        'weather_lookup_key',
        'match_date',
        'latitude_rounded',
        'longitude_rounded',
        'matched_timezone',
        'city_name',
        'country_name',
    ]]
    .drop_duplicates(subset=['weather_lookup_key'])
    .rename(columns={'latitude_rounded': 'latitude', 'longitude_rounded': 'longitude'})
    .sort_values(['match_date', 'country_name', 'city_name'])
    .reset_index(drop=True)
)

weather_lookup_keys.to_csv(lookup_keys_output_path, index=False)

{
    'matches': int(matches['match_id'].nunique()),
    'match_location_rows': len(match_locations),
    'unique_weather_lookup_keys': len(weather_lookup_keys),
    'lookup_problem_rows': len(lookup_problems),
    'lookup_keys_output_path': 'outputs/tables/openmeteo_lookup_keys.csv',
    'min_match_date': weather_lookup_keys['match_date'].min(),
    'max_match_date': weather_lookup_keys['match_date'].max(),
}


{'matches': 314,
 'match_location_rows': 314,
 'unique_weather_lookup_keys': 289,
 'lookup_problem_rows': 0,
 'lookup_keys_output_path': 'outputs/tables/openmeteo_lookup_keys.csv',
 'min_match_date': '2018-06-14',
 'max_match_date': '2024-07-15'}

## Open-Meteo API mit JSONL-Cache abfragen

Der Cache ist eine JSONL-Datei mit einem Datensatz pro Lookup-Key. Beim erneuten Ausfuehren wird diese Datei zuerst geladen. Erfolgreiche Records werden wiederverwendet; fehlgeschlagene oder fehlende Keys werden erneut angefragt.


In [3]:
from pathlib import Path
def load_cached_weather_records(path: Path) -> dict:
    if not path.exists():
        return {}

    records = {}
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            lookup_key = record.get('weather_lookup_key')
            if lookup_key:
                records[lookup_key] = record
    return records


def openmeteo_url(latitude: float, longitude: float, match_date: str) -> str:
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': match_date,
        'end_date': match_date,
        'hourly': ','.join(HOURLY_VARIABLES),
        'timezone': 'auto',
    }
    return f'{OPEN_METEO_ARCHIVE_URL}?{urlencode(params)}'


def fetch_openmeteo_weather(row: pd.Series, pause_seconds: float = 0.25, max_attempts: int = 4) -> dict:
    request_url = openmeteo_url(row['latitude'], row['longitude'], row['match_date'])
    request_headers = {**REQUEST_HEADERS, 'Connection': 'close'}
    payload = None
    status_code = None
    error_message = None
    fetched_at = None

    for attempt in range(1, max_attempts + 1):
        fetched_at = datetime.now(timezone.utc).isoformat()
        try:
            response = requests.get(request_url, headers=request_headers, timeout=(10, 60))
            status_code = response.status_code
            response.raise_for_status()
            payload = response.json()
            error_message = None
            break
        except Exception as exc:
            status_code = getattr(getattr(exc, 'response', None), 'status_code', None)
            payload = None
            error_message = repr(exc)
            if attempt < max_attempts:
                time.sleep(pause_seconds * attempt)

    time.sleep(pause_seconds)
    return {
        'weather_lookup_key': row['weather_lookup_key'],
        'match_date': row['match_date'],
        'latitude': float(row['latitude']),
        'longitude': float(row['longitude']),
        'city_name': row['city_name'],
        'country_name': row['country_name'],
        'request_url': request_url,
        'fetched_at': fetched_at,
        'status_code': status_code,
        'ok': payload is not None,
        'attempts': attempt,
        'error_message': error_message,
        'response': payload,
    }

cached_records = load_cached_weather_records(raw_jsonl_path)
cached_success_keys = {
    key for key, record in cached_records.items()
    if record.get('ok')
}
cached_failed_keys = set(cached_records) - cached_success_keys
keys_to_fetch = [
    key for key in weather_lookup_keys['weather_lookup_key']
    if key not in cached_success_keys
]

with raw_jsonl_path.open('a', encoding='utf-8') as handle:
    for lookup_key in keys_to_fetch:
        row = weather_lookup_keys.loc[weather_lookup_keys['weather_lookup_key'] == lookup_key].iloc[0]
        record = fetch_openmeteo_weather(row)
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()
        cached_records[lookup_key] = record

{
    'cached_success_before_run': len(cached_success_keys),
    'cached_failed_before_run': len(cached_failed_keys),
    'api_requests_this_run': len(keys_to_fetch),
    'cached_success_after_run': sum(1 for record in cached_records.values() if record.get('ok')),
    'cached_records_after_run': len(cached_records),
    'raw_jsonl_path': 'data/raw/weather/openmeteo_raw.jsonl',
}


{'cached_success_before_run': 289,
 'cached_failed_before_run': 0,
 'api_requests_this_run': 0,
 'cached_success_after_run': 289,
 'cached_records_after_run': 289,
 'raw_jsonl_path': 'data/raw/weather/openmeteo_raw.jsonl'}

## API-Antworten in strukturierte Beobachtungen transformieren

Open-Meteo liefert Stuendenwerte. Fuer jedes Match wird die Wetterzeile gewaehlt, deren Zeitstempel dem Kickoff am naechsten liegt. Die Bronze-Tabelle bleibt weiterhin match-level, damit Kafka- und Spark-Schritte die Wetterdaten strukturiert weiterverarbeiten koennen.


In [4]:
def weather_record_to_hourly_frame(record: dict) -> pd.DataFrame:
    if not record.get('ok') or not record.get('response'):
        return pd.DataFrame()

    response = record['response']
    hourly = response.get('hourly') or {}
    times = hourly.get('time') or []
    if not times:
        return pd.DataFrame()

    frame = pd.DataFrame({'weather_time': pd.to_datetime(times)})
    for variable in HOURLY_VARIABLES:
        values = hourly.get(variable)
        frame[variable] = values if values is not None else pd.NA

    frame['weather_lookup_key'] = record['weather_lookup_key']
    frame['openmeteo_timezone'] = response.get('timezone')
    frame['openmeteo_utc_offset_seconds'] = response.get('utc_offset_seconds')
    frame['openmeteo_elevation'] = response.get('elevation')
    frame['request_url'] = record.get('request_url')
    return frame

hourly_weather_columns = [
    'weather_time',
    *HOURLY_VARIABLES,
    'weather_lookup_key',
    'openmeteo_timezone',
    'openmeteo_utc_offset_seconds',
    'openmeteo_elevation',
    'request_url',
]
hourly_weather = pd.concat(
    [weather_record_to_hourly_frame(record) for record in cached_records.values()],
    ignore_index=True,
) if cached_records else pd.DataFrame(columns=hourly_weather_columns)
if hourly_weather.empty:
    hourly_weather = pd.DataFrame(columns=hourly_weather_columns)

hourly_weather.to_parquet(hourly_weather_output_path, index=False)

weather_observation_columns = [
    'match_id',
    'match_date',
    'kick_off',
    'kickoff_timestamp',
    'weather_time',
    'hours_from_kickoff',
    'weather_lookup_key',
    'stadium_id',
    'stadium_name',
    'city_name',
    'country_name',
    'latitude',
    'longitude',
    'openmeteo_timezone',
    'openmeteo_utc_offset_seconds',
    'openmeteo_elevation',
    'temperature_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'request_url',
    'short_name',
    'competition_name',
    'season_name',
]
candidate_columns = list(match_locations.columns) + [
    column for column in hourly_weather_columns if column != 'weather_lookup_key'
] + ['hours_from_kickoff']
candidate_rows = pd.DataFrame(columns=candidate_columns)
missing_weather_matches = pd.DataFrame()

if hourly_weather.empty:
    weather_observations = pd.DataFrame(columns=weather_observation_columns)
    missing_weather_matches = match_locations.loc[~lookup_problem_mask].copy()
    missing_weather_matches['request_url'] = None
else:
    candidate_rows = match_locations.loc[~lookup_problem_mask].merge(
        hourly_weather,
        on='weather_lookup_key',
        how='left',
        validate='many_to_many',
    )
    candidate_rows['hours_from_kickoff'] = (
        candidate_rows['weather_time'] - candidate_rows['kickoff_timestamp']
    ).abs() / pd.Timedelta(hours=1)

    missing_weather_matches = (
        candidate_rows.loc[candidate_rows['weather_time'].isna()]
        .drop_duplicates(subset=['match_id'])
        .copy()
    )
    valid_candidate_rows = candidate_rows.dropna(subset=['hours_from_kickoff'])

    if valid_candidate_rows.empty:
        weather_observations = pd.DataFrame(columns=weather_observation_columns)
    else:
        nearest_index = valid_candidate_rows.groupby('match_id')['hours_from_kickoff'].idxmin()
        weather_observations = valid_candidate_rows.loc[nearest_index].copy()

        weather_observations = weather_observations[weather_observation_columns].sort_values(
            ['match_date', 'match_id']
        ).reset_index(drop=True)

imputation_rows = []
if not weather_observations.empty:
    rain_fallback_mask = weather_observations['rain'].isna() & weather_observations['precipitation'].notna()
    for _, row in weather_observations.loc[rain_fallback_mask].iterrows():
        imputation_rows.append({
            'match_id': row['match_id'],
            'weather_lookup_key': row['weather_lookup_key'],
            'weather_time': row['weather_time'],
            'column': 'rain',
            'replacement_value': row['precipitation'],
            'fallback_source': 'precipitation',
            'reason': 'Open-Meteo rain value missing while precipitation is available.',
        })
    weather_observations.loc[rain_fallback_mask, 'rain'] = weather_observations.loc[
        rain_fallback_mask, 'precipitation'
    ]

    apparent_temp_fallback_mask = (
        weather_observations['apparent_temperature'].isna()
        & weather_observations['temperature_2m'].notna()
    )
    for _, row in weather_observations.loc[apparent_temp_fallback_mask].iterrows():
        imputation_rows.append({
            'match_id': row['match_id'],
            'weather_lookup_key': row['weather_lookup_key'],
            'weather_time': row['weather_time'],
            'column': 'apparent_temperature',
            'replacement_value': row['temperature_2m'],
            'fallback_source': 'temperature_2m',
            'reason': 'Open-Meteo apparent_temperature value missing while temperature_2m is available.',
        })
    weather_observations.loc[
        apparent_temp_fallback_mask, 'apparent_temperature'
    ] = weather_observations.loc[apparent_temp_fallback_mask, 'temperature_2m']

weather_imputations = pd.DataFrame(imputation_rows)
if weather_imputations.empty:
    weather_imputations = pd.DataFrame(columns=[
        'match_id', 'weather_lookup_key', 'weather_time', 'column',
        'replacement_value', 'fallback_source', 'reason'
    ])
weather_imputations.to_csv(weather_imputation_output_path, index=False)

candidate_rows.to_parquet(
    candidate_rows_output_path,
    index=False,
    coerce_timestamps='us',
    allow_truncated_timestamps=True,
)
missing_weather_matches.to_csv(missing_weather_output_path, index=False)

{
    'hourly_weather_rows': len(hourly_weather),
    'candidate_rows': len(candidate_rows),
    'weather_rows': len(weather_observations),
    'missing_weather_matches': len(missing_weather_matches),
    'weather_imputations': len(weather_imputations),
    'hourly_weather_output_path': 'data/cache/weather/openmeteo_hourly_weather.parquet',
    'candidate_rows_output_path': 'data/cache/weather/openmeteo_candidate_rows.parquet',
    'missing_weather_output_path': 'outputs/tables/openmeteo_missing_weather_matches.csv',
    'weather_imputation_output_path': 'outputs/tables/openmeteo_weather_imputations.csv',
}


{'hourly_weather_rows': 6936,
 'candidate_rows': 7536,
 'weather_rows': 314,
 'missing_weather_matches': 0,
 'weather_imputations': 118,
 'hourly_weather_output_path': 'data/cache/weather/openmeteo_hourly_weather.parquet',
 'candidate_rows_output_path': 'data/cache/weather/openmeteo_candidate_rows.parquet',
 'missing_weather_output_path': 'outputs/tables/openmeteo_missing_weather_matches.csv',
 'weather_imputation_output_path': 'outputs/tables/openmeteo_weather_imputations.csv'}

## Fehler dokumentieren und Outputs speichern

Fehler werden aus drei Quellen gesammelt: fehlende Join-/Geocode-Informationen, fehlgeschlagene API-Responses und Matches ohne passende Stuendenwerte. Diese Tabelle erfuellt das BD-11-Kriterium, dass problematische API-Abfragen sichtbar dokumentiert werden.


In [5]:
failure_rows = []

for _, row in lookup_problems.iterrows():
    missing_fields = [
        column for column in ['match_date', 'kickoff_timestamp', 'latitude', 'longitude']
        if pd.isna(row[column])
    ]
    failure_rows.append({
        'failure_type': 'missing_lookup_input',
        'match_id': row.get('match_id'),
        'weather_lookup_key': row.get('weather_lookup_key'),
        'match_date': row.get('match_date'),
        'city_name': row.get('city_name'),
        'country_name': row.get('country_name'),
        'request_url': None,
        'status_code': None,
        'details': ', '.join(missing_fields),
    })

for record in cached_records.values():
    if not record.get('ok'):
        failure_rows.append({
            'failure_type': 'api_request_failed',
            'match_id': None,
            'weather_lookup_key': record.get('weather_lookup_key'),
            'match_date': record.get('match_date'),
            'city_name': record.get('city_name'),
            'country_name': record.get('country_name'),
            'request_url': record.get('request_url'),
            'status_code': record.get('status_code'),
            'details': record.get('error_message'),
        })

for _, row in missing_weather_matches.iterrows():
    failure_rows.append({
        'failure_type': 'missing_hourly_weather_values',
        'match_id': row.get('match_id'),
        'weather_lookup_key': row.get('weather_lookup_key'),
        'match_date': row.get('match_date'),
        'city_name': row.get('city_name'),
        'country_name': row.get('country_name'),
        'request_url': row.get('request_url'),
        'status_code': None,
        'details': 'No hourly weather row returned for lookup key.',
    })

failures = pd.DataFrame(failure_rows)
if failures.empty:
    failures = pd.DataFrame(columns=[
        'failure_type', 'match_id', 'weather_lookup_key', 'match_date', 'city_name',
        'country_name', 'request_url', 'status_code', 'details'
    ])

weather_observations.to_parquet(
    weather_output_path,
    index=False,
    coerce_timestamps='us',
    allow_truncated_timestamps=True,
)
failures.to_csv(failure_output_path, index=False)

{
    'weather_rows': len(weather_observations),
    'unique_matches_with_weather': int(weather_observations['match_id'].nunique()) if not weather_observations.empty else 0,
    'failure_rows': len(failures),
    'weather_output_path': 'data/bronze/weather_observations.parquet',
    'failure_output_path': 'outputs/tables/openmeteo_failures.csv',
}


{'weather_rows': 314,
 'unique_matches_with_weather': 314,
 'failure_rows': 0,
 'weather_output_path': 'data/bronze/weather_observations.parquet',
 'failure_output_path': 'outputs/tables/openmeteo_failures.csv'}

## Qualitaetschecks

Die Checks bilden die BD-11-Akzeptanzkriterien ab: Wetterdaten muessen fuer alle Match-Orte und Matchdaten vorhanden sein, Lookup-Keys duerfen nicht mehrfach geschrieben werden, und die benoetigten Wettervariablen muessen gefuellt sein.


In [6]:
required_weather_columns = ['temperature_2m', 'apparent_temperature', 'precipitation', 'rain']
expected_match_count = matches['match_id'].nunique()
observed_match_count = weather_observations['match_id'].nunique() if not weather_observations.empty else 0
failure_type_counts = failures['failure_type'].value_counts().to_dict() if not failures.empty else {}
imputation_counts = weather_imputations['column'].value_counts().to_dict() if not weather_imputations.empty else {}
missing_match_count = expected_match_count - observed_match_count

quality_summary = {
    'expected_matches': int(expected_match_count),
    'matches_with_weather': int(observed_match_count),
    'missing_matches': int(missing_match_count),
    'failure_rows': len(failures),
    'failure_type_counts': failure_type_counts,
    'imputation_counts': imputation_counts,
    'failure_output_path': 'outputs/tables/openmeteo_failures.csv',
    'weather_imputation_output_path': 'outputs/tables/openmeteo_weather_imputations.csv',
}
display(quality_summary)

missing_weather_columns = sorted(set(required_weather_columns) - set(weather_observations.columns))
assert not missing_weather_columns, f'Missing weather columns: {missing_weather_columns}'
assert weather_lookup_keys['weather_lookup_key'].is_unique, 'Weather lookup keys should be unique before API calls.'
assert weather_observations['match_id'].is_unique, 'Each match should have exactly one weather observation.'
assert failures.empty, (
    f'Open-Meteo failures were documented in {failure_output_path}: {failure_type_counts}. '
    'Re-run from the API cache cell; failed keys are retried with multiple attempts per run.'
)
assert len(weather_observations) == expected_match_count, (
    'Weather observations should contain one row per match. '
    f'Expected {expected_match_count}, got {len(weather_observations)}.'
)
assert weather_observations[required_weather_columns].notna().all().all(), 'Required weather values must not be missing.'

{
    'matches_with_weather': int(weather_observations['match_id'].nunique()),
    'temperature_min': float(weather_observations['temperature_2m'].min()),
    'temperature_max': float(weather_observations['temperature_2m'].max()),
    'apparent_temperature_min': float(weather_observations['apparent_temperature'].min()),
    'apparent_temperature_max': float(weather_observations['apparent_temperature'].max()),
    'precipitation_total': float(weather_observations['precipitation'].sum()),
    'rain_total': float(weather_observations['rain'].sum()),
}


{'expected_matches': 314,
 'matches_with_weather': 314,
 'missing_matches': 0,
 'failure_rows': 0,
 'failure_type_counts': {},
 'imputation_counts': {'rain': 66, 'apparent_temperature': 52},
 'failure_output_path': 'outputs/tables/openmeteo_failures.csv',
 'weather_imputation_output_path': 'outputs/tables/openmeteo_weather_imputations.csv'}

{'matches_with_weather': 314,
 'temperature_min': 12.2,
 'temperature_max': 38.4,
 'apparent_temperature_min': 7.6,
 'apparent_temperature_max': 37.5,
 'precipitation_total': 25.5,
 'rain_total': 25.5}